In [69]:
import pandas as pd
import numpy as np

In [70]:
df = pd.read_csv('university_raw_data.csv')
print(f"Original row count: {len(df)}")

Original row count: 3467


In [71]:
df.shape

(3467, 54)

In [72]:
df.head()

,2024 RANK,2023 RANK,Institution Name,Country Code,Country,SIZE,FOCUS,RES.,AGE,STATUS,...,location,stats_number_students,stats_student_staff_ratio,stats_pc_intl_students,stats_female_male_ratio,stats_proportion_of_isr,aliases,subjects_offered,closed,unaccredited
0,1201-1400,1201-1400,Don State Technical University,RU,Russia,L,CO,HI,4,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,441,354,University of Dundee,UK,United Kingdom,L,FC,HI,4,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,375,501-510,Western Sydney University,AU,Australia,XL,FC,HI,3,A,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,United Kingdom,780,19.5,39%,45:55:00,23%,AECC University College,"Sport Science,Psychology,Other Health",False,False
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Poland,"19,189",12.2,3%,35 : 65,49%,AGH University of Krakow,"Mechanical & Aerospace Engineering,Chemical En...",False,False


In [73]:
df.dropna(how='all', inplace=True)

In [74]:
df = df[df['Institution Name'] != 'institution']

In [75]:
df['University_Name'] = df['Institution Name'].combine_first(df['name'])

In [76]:
print(f"Row count after initial cleanup: {len(df)}")

Row count after initial cleanup: 3466


Cleaning

In [77]:
# Clean ranking symbols ('=' and '+')
print("Removing '+' and '=' symbols...")
df = df.replace({r'\+': '', '=': ''}, regex=True)

# columns with (+ and =) symbols
columns_to_check = [
    'Academic Reputation Rank',
    'Employer Reputation Rank',
    'Faculty Student Rank',
    'International Faculty Rank',
    'International Students Rank',
    'International Research Network Rank',
    'Employment Outcomes Rank',
    'Sustainability Rank',
    'rank'
]

display(df[columns_to_check].dropna(how='all').head(10))

Removing '+' and '=' symbols...


,Academic Reputation Rank,Employer Reputation Rank,Faculty Student Rank,International Faculty Rank,International Students Rank,International Research Network Rank,Employment Outcomes Rank,Sustainability Rank,rank
0,601,601,701,701,624,701,701,701,NaN
1,488,595,418,142,162,475,701,524,NaN
2,483,546,701,48,351,206,701,92,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Reporter
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001–1200
5,601,542,422,701,701,623,701,701,NaN
6,442,424,500,244,570,151,569,393,201–250
7,187,223,391,147,363,277,39,53,201–250
8,140,387,490,181,661,56,156,18,109
9,372,510,327,423,701,701,701,701,NaN


midpoint of ranks

In [78]:

print("Converting rank ranges to midpoints and handling text...")

# 1. handle the ranges and 'Reporter' text
def clean_rank_values(val):
    if pd.isna(val):
        return val
    
    val = str(val).strip()
    
    # Reporter to Null/NaN value
    if val.lower() == 'reporter':
        return np.nan
    
    # calculate the average midpoint
    if '-' in val or '–' in val:
        val = val.replace('–', '-') # Standardize the dash
        parts = val.split('-')
        return (float(parts[0]) + float(parts[1])) / 2
        
    return float(val)

# 2. Apply this function specifically to the THE 'rank' column
df['rank'] = df['rank'].apply(clean_rank_values)

# 3. rank columns to numerical data types (Floats)
for col in columns_to_check:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types successfully converted to numerical format!")


display(df[columns_to_check].dropna(how='all').head(15))

Converting rank ranges to midpoints and handling text...
Data types successfully converted to numerical format!


,Academic Reputation Rank,Employer Reputation Rank,Faculty Student Rank,International Faculty Rank,International Students Rank,International Research Network Rank,Employment Outcomes Rank,Sustainability Rank,rank
0,601.0,601.0,701.0,701.0,624.0,701.0,701.0,701.0,NaN
1,488.0,595.0,418.0,142.0,162.0,475.0,701.0,524.0,NaN
2,483.0,546.0,701.0,48.0,351.0,206.0,701.0,92.0,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1100.5
5,601.0,542.0,422.0,701.0,701.0,623.0,701.0,701.0,NaN
6,442.0,424.0,500.0,244.0,570.0,151.0,569.0,393.0,225.5
7,187.0,223.0,391.0,147.0,363.0,277.0,39.0,53.0,225.5
8,140.0,387.0,490.0,181.0,661.0,56.0,156.0,18.0,109.0
9,372.0,510.0,327.0,423.0,701.0,701.0,701.0,701.0,NaN
13,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1501.0


The Missing Values showcase

In [79]:
print("Auditing missing values...")

# Calculate % of missing values
missing_data = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({'Missing_Percentage': missing_data})
missing_df = missing_df.sort_values(by='Missing_Percentage', ascending=False)

pd.set_option('display.max_rows', None) 

print("The complete missing values audit:")
display(missing_df)

pd.reset_option('display.max_rows')

Auditing missing values...
The complete missing values audit:


,Missing_Percentage
International Faculty Rank,60.444316
International Faculty Score,60.444316
Sustainability Rank,59.694172
Sustainability Score,59.694172
RES.,59.319100
2023 RANK,59.174841
International Students Rank,59.117138
International Students Score,59.117138
STATUS,58.020773
AGE,57.559146


In [80]:
overall_missing = (df.isnull().sum().sum() / df.size) * 100
print(f"{overall_missing:.2f}%")

43.89%


droping unnecessary columns  

In [81]:
print("Dropping unnecessary columns...")

columns_to_drop = [
    'Institution Name', 
    'name', 
    'aliases', 
    'closed', 
    'unaccredited', 
    'Country Code'
]

df = df.drop(columns=columns_to_drop)

print(f"Columns successfully dropped. Total remaining columns: {len(df.columns)}")

Dropping unnecessary columns...
Columns successfully dropped. Total remaining columns: 49


In [82]:
overall_missing = (df.isnull().sum().sum() / df.size) * 100
print(f"{overall_missing:.2f}%")

45.08%


merging country and location's data to reduce the missing %

In [83]:
print("Executing categorical data rescue and imputation...")

df['Master_Country'] = df['Country'].combine_first(df['location'])

df = df.drop(columns=['Country', 'location'])

missing_country_pct = (df['Master_Country'].isnull().sum() / len(df)) * 100

print(f"Old Country missing rate: ~56.81%")
print(f"New Master_Country missing rate: {missing_country_pct:.2f}%")

display(df[['Master_Country', 'SIZE', 'FOCUS', 'STATUS']].head(10))

Executing categorical data rescue and imputation...
Old Country missing rate: ~56.81%
New Master_Country missing rate: 0.03%


,Master_Country,SIZE,FOCUS,STATUS
0,Russia,L,CO,A
1,United Kingdom,L,FC,A
2,Australia,XL,FC,A
3,United Kingdom,NaN,NaN,NaN
4,Poland,NaN,NaN,NaN
5,Poland,L,CO,A
6,Denmark,L,FC,A
7,Finland,L,FO,A
8,Denmark,L,FC,A
9,Kazakhstan,L,CO,A


In [84]:
overall_missing = (df.isnull().sum().sum() / df.size) * 100
print(f"{overall_missing:.2f}%")

44.35%


filling empty columns with 'Unknown'

In [85]:
categorical_cols_to_fill = ['SIZE', 'FOCUS', 'STATUS']
df[categorical_cols_to_fill] = df[categorical_cols_to_fill].fillna('Unknown')

In [86]:
print("Verifying categorical column integrity...")

# Check the missing percentage for SIZE, FOCUS, and STATUS
verification_cols = ['SIZE', 'FOCUS', 'STATUS']
verification_check = (df[verification_cols].isnull().sum() / len(df)) * 100

display(verification_check)

Verifying categorical column integrity...


SIZE      0.0
FOCUS     0.0
STATUS    0.0
dtype: float64

In [87]:
overall_missing = (df.isnull().sum().sum() / df.size) * 100
print(f"{overall_missing:.2f}%")

40.76%


cleaning stats columns

In [88]:
print("Transforming statistical text columns into clean numbers...")

# 1. remove % symbols
pct_columns = ['stats_pc_intl_students', 'stats_proportion_of_isr']
for col in pct_columns:
    df[col] = df[col].astype(str).str.replace('%', '', regex=False)

    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Transform 'stats_female_male_ratio' into a single numeric percentage
def extract_female_ratio(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    
    # If the string contains a colon, split it and grab the first part
    if ':' in val:
        parts = val.split(':')
        return float(parts[0].strip())
        
    return np.nan


df['stats_female_male_ratio'] = df['stats_female_male_ratio'].apply(extract_female_ratio)

print("Transformations complete! Here is the clean numerical data:")
display(df[['stats_pc_intl_students', 'stats_proportion_of_isr', 'stats_female_male_ratio']].head(15))

Transforming statistical text columns into clean numbers...
Transformations complete! Here is the clean numerical data:


,stats_pc_intl_students,stats_proportion_of_isr,stats_female_male_ratio
0,NaN,NaN,NaN
1,NaN,NaN,NaN
2,NaN,NaN,NaN
3,39.0,23.0,45.0
4,3.0,49.0,35.0
5,NaN,NaN,NaN
6,12.0,34.0,NaN
7,23.0,46.0,39.0
8,9.0,25.0,56.0
9,NaN,NaN,NaN


In [89]:
print("Fixing commas and finalizing Group 1 demographics...")

df = df.rename(columns={'stats_female_male_ratio': 'female_student_pct'})

# Define our Group 1 columns (The Demographics/Counts)
group_1_demographics = [
    'stats_number_students', 
    'stats_student_staff_ratio', 
    'stats_pc_intl_students', 
    'stats_proportion_of_isr',
    'female_student_pct'
]

# removing commas
for col in group_1_demographics:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(',', '', regex=False)
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Fill missing values with  Median 
for col in group_1_demographics:
    
    col_median = df[col].median()

    df[col] = df[col].fillna(col_median)

print("Demographics successfully cleaned, renamed, and imputed!")

display((df[group_1_demographics].isnull().sum() / len(df)) * 100)

Fixing commas and finalizing Group 1 demographics...
Demographics successfully cleaned, renamed, and imputed!


stats_number_students        0.0
stats_student_staff_ratio    0.0
stats_pc_intl_students       0.0
stats_proportion_of_isr      0.0
female_student_pct           0.0
dtype: float64

In [90]:
overall_missing = (df.isnull().sum().sum() / df.size) * 100
print(f"{overall_missing:.2f}%")

38.19%


filling zeros for scores columns

In [91]:
print("Executing Zero-Fill for Group 2 Performance Scores...")

group_2_scores = [
    'Academic Reputation Score', 
    'Employer Reputation Score', 
    'Faculty Student Score', 
    'Citations per Faculty Score', 
    'International Faculty Score', 
    'International Students Score', 
    'International Research Network Score', 
    'Employment Outcomes Score', 
    'Sustainability Score', 
    'Overall SCORE', 
    'scores_overall', 
    'scores_teaching', 
    'scores_research', 
    'scores_citations', 
    'scores_industry_income', 
    'scores_international_outlook'
]

#  convert to numeric  and fill the missing gaps with 0
for col in group_2_scores:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = df[col].fillna(0)

print("successfully imputed  0!")

score_missing_pct = (df[group_2_scores].isnull().sum() / len(df)) * 100
display(score_missing_pct)

Executing Zero-Fill for Group 2 Performance Scores...
successfully imputed  0!


Academic Reputation Score               0.0
Employer Reputation Score               0.0
Faculty Student Score                   0.0
Citations per Faculty Score             0.0
International Faculty Score             0.0
International Students Score            0.0
International Research Network Score    0.0
Employment Outcomes Score               0.0
Sustainability Score                    0.0
Overall SCORE                           0.0
scores_overall                          0.0
scores_teaching                         0.0
scores_research                         0.0
scores_citations                        0.0
scores_industry_income                  0.0
scores_international_outlook            0.0
dtype: float64

filling unknown

In [92]:
print("Executing text data imputation...")

pending_text_cols = ['RES.', 'AGE', 'subjects_offered']

df[pending_text_cols] = df[pending_text_cols].fillna('Unknown')

print("successfully labeled as Unknown!")

display((df[pending_text_cols].isnull().sum() / len(df)) * 100)

Executing text data imputation...
successfully labeled as Unknown!


RES.                0.0
AGE                 0.0
subjects_offered    0.0
dtype: float64

Analyzing Rank Columns

In [93]:
print("Analyzing missing data in Group 3 (Rank Columns)...")

rank_columns = [
    '2024 RANK', '2023 RANK', 'rank_order', 'rank', 
    'Academic Reputation Rank', 'Employer Reputation Rank', 
    'Faculty Student Rank', 'Citations per Faculty Rank', 
    'International Faculty Rank', 'International Students Rank', 
    'International Research Network Rank', 'Employment Outcomes Rank', 
    'Sustainability Rank', 'scores_overall_rank', 'scores_teaching_rank', 
    'scores_research_rank', 'scores_citations_rank', 
    'scores_industry_income_rank', 'scores_international_outlook_rank'
]

rank_missing_pct = (df[rank_columns].isnull().sum() / len(df)) * 100

display(rank_missing_pct)

Analyzing missing data in Group 3 (Rank Columns)...


2024 RANK                              56.809002
2023 RANK                              59.174841
rank_order                             22.937103
rank                                   45.008656
Academic Reputation Rank               56.809002
Employer Reputation Rank               56.837853
Faculty Student Rank                   57.501443
Citations per Faculty Rank             57.501443
International Faculty Rank             60.444316
International Students Rank            59.117138
International Research Network Rank    56.924409
Employment Outcomes Rank               57.501443
Sustainability Rank                    59.694172
scores_overall_rank                    22.937103
scores_teaching_rank                   22.937103
scores_research_rank                   22.937103
scores_citations_rank                  22.937103
scores_industry_income_rank            22.937103
scores_international_outlook_rank      22.937103
dtype: float64

In [94]:

print("Cleaning banded ranges and symbols from Rank columns...")

def clean_banded_ranks(val):
    if pd.isna(val):
        return np.nan
    
    val = str(val).strip()
    
    # Remove the '=' signs
    if '=' in val:
        val = val.replace('=', '')
        
    #  calculate midpoint for ranges
    if '-' in val:
        parts = val.split('-')
        try:
            return (float(parts[0].strip()) + float(parts[1].strip())) / 2
        except ValueError:
            return np.nan # Failsafe
            
    # convert string to float
    try:
        return float(val)
    except ValueError:
        return np.nan

cols_to_clean = ['2024 RANK', '2023 RANK']
for col in cols_to_clean:
    df[col] = df[col].apply(clean_banded_ranks)

print("Rank cleaning successful!")
display(df[['2024 RANK', '2023 RANK']].head(15))

Cleaning banded ranges and symbols from Rank columns...
Rank cleaning successful!


,2024 RANK,2023 RANK
0,1300.5,1300.5
1,441.0,354.0
2,375.0,505.5
3,NaN,NaN
4,NaN,NaN
5,925.5,900.5
6,336.0,330.0
7,109.0,116.0
8,143.0,161.0
9,685.5,515.5


In [95]:
overall_missing = (df.isnull().sum().sum() / df.size) * 100
print(f"{overall_missing:.2f}%")

17.58%


In [96]:
print("Auditing missing values...")

# Calculate % of missing values
missing_data = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({'Missing_Percentage': missing_data})
missing_df = missing_df.sort_values(by='Missing_Percentage', ascending=False)

print("Top 20 columns with the highest missing values:")
display(missing_df.head(20))

Auditing missing values...
Top 20 columns with the highest missing values:


,Missing_Percentage
International Faculty Rank,60.444316
Sustainability Rank,59.694172
2023 RANK,59.174841
International Students Rank,59.117138
Employment Outcomes Rank,57.501443
Citations per Faculty Rank,57.501443
Faculty Student Rank,57.501443
International Research Network Rank,56.924409
Employer Reputation Rank,56.837853
2024 RANK,56.809002


In [97]:
all_target_columns = [
    'International Faculty Rank', 'Sustainability Rank', '2023 RANK', 
    'International Students Rank', 'Employment Outcomes Rank', 
    'Citations per Faculty Rank', 'Faculty Student Rank', 
    'International Research Network Rank', 'Employer Reputation Rank', 
    '2024 RANK', 'Academic Reputation Rank', 'rank',
    'scores_overall_rank', 'rank_order', 'scores_international_outlook_rank', 
    'scores_industry_income_rank', 'scores_citations_rank', 
    'scores_research_rank', 'scores_teaching_rank', 'Master_Country'
]

df_final = df.dropna(subset=all_target_columns)

print(f"Final Row Count: {len(df_final)}")
print(f"Final Column Count: {df_final.shape[1]}")


Final Row Count: 620
Final Column Count: 48


In [98]:
total_missing = df_final.isnull().sum().sum()
total_cells = df_final.size

overall_missing_percentage = (total_missing / total_cells) * 100

print(f"Total Missing Values: {total_missing}")
print(f"Total Cells: {total_cells}")
print(f"Overall Missing Percentage: {overall_missing_percentage:.2f}%")

Total Missing Values: 0
Total Cells: 29760
Overall Missing Percentage: 0.00%


In [99]:
print("--- Top Columns with Missing Values ---")

missing_percentages = (df_final.isnull().sum() / len(df_final)) * 100
missing_percentages = missing_percentages[missing_percentages > 0]
missing_percentages = missing_percentages.sort_values(ascending=False)

if missing_percentages.empty:
    print("No missing values found! All columns are 100% complete.")
else:
    print(missing_percentages)

--- Top Columns with Missing Values ---
No missing values found! All columns are 100% complete.


In [100]:
df_final = pd.read_csv('cleaned_university_data_2024.csv')

df_final = df_final.dropna(subset=['scores_overall', 'Overall SCORE', 'RES.'])

total_missing = df_final.isnull().sum().sum()
total_cells = df_final.size
overall_missing_percentage = (total_missing / total_cells) * 100

print(f"Final Row Count: {len(df_final)}")
print(f"Total Missing Values: {total_missing}")
print(f"Overall Missing Percentage: {overall_missing_percentage:.2f}%")

df_final.to_csv('cleaned_university_data_2024.csv', index=False)
print("File successfully updated and saved.")

Final Row Count: 620
Total Missing Values: 0
Overall Missing Percentage: 0.00%


File successfully updated and saved.
